# Pilot: sampling-based MPC (MPPI) expert for Push-T

**Goal:** measure, before committing compute, how fast a sampling-based planner runs on a Colab GPU and how often it succeeds,
so we can decide whether to use it as the non-RL demonstration source for the BC teacher (v2).

**What the expert does.** At every control step it copies the current simulator state into N parallel copies, tries N smooth
candidate stick-tip motions over the next H steps in the simulator itself, scores each by how close the T ends up to the goal
pose (plus small terms for staying near the T and effort; this is not PPO's reward), averages the candidates weighted by score,
executes the first step, and repeats. No learned policy and no PPO involvement.

**This pilot runs** three search budgets (N × H = 128 × 8, 256 × 10, 512 × 12), 32 episodes each (8 per goal quadrant, 100
steps, the team's validation seeds), and reports success-once, success-at-end, measured seconds per episode, and the extrapolated
cost of 4,096 and 16,384 demonstrations. It then renders top-down videos of the best setting.

Select **Runtime → Change runtime type → L4 GPU**. Upload only this notebook. Expected time: about 20–40 minutes.

In [ ]:
from pathlib import Path
import os, sys, json, re, subprocess, datetime, shlex, shutil, base64, hashlib

MANISKILL_COMMIT = "a4a4f9272ad64b1564035874b605ceb687b63ed8"  # v3.0.1, same as the teacher notebooks
MS_ROOT = Path("/content/ManiSkill")
WORK_ROOT = Path("/content/pusht_state_bc")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BASE = Path("/content/drive/MyDrive/182_final_project/state_bc")
RUN_LABEL = "mppi_pilot"
JIMMY_COMMIT = "c87d8ae3669c10894d765df2d0dd4b61c010e272"

EPISODES_PER_BATCH = 8           # episodes run in lockstep; each gets N simulator copies
CONFIGS = [(128, 8), (256, 10), (512, 12)]   # (samples N, horizon H)
QUADRANTS = [1, 2, 3, 4]

SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + RUN_LABEL
subprocess.run(["nvidia-smi"], check=True)

## 1. Drive and logging

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTPUT_ROOT = DRIVE_BASE / SESSION
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
LOG = OUTPUT_ROOT / "notebook.log"
print("Session folder:", OUTPUT_ROOT)


def run(command, show=None, **kwargs):
    """Run a command and append all output to notebook.log. Prints every line, or only lines
    matching the regex `show`. On failure, raises with the last 40 lines."""
    command = list(map(str, command))
    print("$", shlex.join(command), flush=True)
    pattern = re.compile(show) if show else None
    tail = []
    with LOG.open("a") as log:
        log.write("\n$ " + shlex.join(command) + "\n")
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            try:
                for line in process.stdout:
                    if pattern is None or pattern.search(line):
                        print(line, end="", flush=True)
                    log.write(line)
                    tail = (tail + [line.rstrip()])[-40:]
                code = process.wait()
            except BaseException:
                process.kill()
                raise
    if code:
        raise RuntimeError(f"Command failed (exit {code}). Log: {LOG}\n" + "\n".join(tail))

## 2. Install ManiSkill v3.0.1 under Python 3.11 (same as the other notebooks)

In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libvulkan1", "libvulkan-dev", "vulkan-tools", "libglvnd-dev"])
if not (MS_ROOT / ".git").exists():
    run(["git", "clone", "https://github.com/mani-skill/ManiSkill.git", MS_ROOT])
run(["git", "-C", MS_ROOT, "checkout", "--detach", MANISKILL_COMMIT])
assert subprocess.check_output(["git", "-C", str(MS_ROOT), "rev-parse", "HEAD"], text=True).strip() == MANISKILL_COMMIT

run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.11"])
TRAIN_ENV = WORK_ROOT / "python311"
TRAIN_PY = TRAIN_ENV / "bin/python"
if not TRAIN_PY.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", TRAIN_ENV])
constraints = WORK_ROOT / "constraints.txt"
constraints.write_text("numpy==1.26.4\nopencv-python==4.11.0.86\nsapien==3.0.3\ntorch==2.6.0\n")
run([TRAIN_PY, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
run([TRAIN_PY, "-m", "pip", "install", "-q", "--constraint", constraints, "-e", MS_ROOT,
     "numpy", "sapien", "h5py", "tyro", "imageio[ffmpeg]", "matplotlib"])

for filename, library, vulkan in [
    ("/usr/share/vulkan/icd.d/nvidia_icd.json", "libGLX_nvidia.so.0", True),
    ("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "libEGL_nvidia.so.0", False),
]:
    path = Path(filename)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        icd = {"library_path": library}
        if vulkan:
            icd["api_version"] = "1.2.155"
        path.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": icd}))
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
os.environ["MS_SKIP_ASSET_DOWNLOAD_PROMPT"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
run([TRAIN_PY, "-c", "import torch, sapien, mani_skill, pytorch_kinematics; print('torch', torch.__version__, 'mani_skill', "
     "mani_skill.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"])

## 3. Write the modules

`mppi_gpu.py` is this repo's `state_bc/scripted_pusher/mppi/mppi_gpu.py`. `pusht_vargoal.py` and its arm-pose library are
copied unchanged from Jimmy's branch at `c87d8ae`, checked by SHA-256.

In [ ]:
%%writefile /content/pusht_state_bc/mppi_gpu.py
"""Batched sampling-based MPC (MPPI) expert for PushTVarGoal-v1, for the GPU simulator.

E episodes run in lockstep inside one ManiSkill environment with E*N parallel copies: copy e*N is episode e's "leader"
(the real episode), copies e*N+1 .. e*N+N-1 are its scratch copies for rollouts. Every control step:
  1. copy each leader's full simulator state into its N copies;
  2. roll out N candidate tip-velocity plans per episode for H steps (all E*N in parallel), converting tip velocities to
     joint-delta actions with batched damped-least-squares IK;
  3. cost = T pose error to the goal at the end (+ small tip-to-T distance and effort terms): a plain geometric cost, not
     PPO's reward; MPPI-weight the N candidates of each episode into a new plan;
  4. restore every copy to its leader's state and apply the plan's first action to all copies of the group (so all copies,
     including the leader, take the real step), then shift the plan.
The executed joint-delta action is recorded as the BC label, with the 31-number state in PushT-v1's `state` layout.
Optional action noise on the executed step (DART-style: record the clean action, execute a noisy one).

Usage (from the pilot notebook):
    python mppi_gpu.py --episodes-per-batch 8 --samples 256 --horizon 10 --quadrants 1 2 3 4 --out OUT
"""
from __future__ import annotations

import argparse
import json
import math
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parent))

GOAL_THETA = 5.0 / 3.0 * math.pi
JOINT_DELTA_SCALE = 0.1
DT = 1.0 / 20.0
T_POLY = torch.tensor([(-0.1, -0.0625), (0.1, -0.0625), (0.1, -0.0125), (0.025, -0.0125),
                       (0.025, 0.1375), (-0.025, 0.1375), (-0.025, -0.0125), (-0.1, -0.0125)])


@dataclass
class MPPIConfig:
    samples: int = 256
    horizon: int = 10
    temperature: float = 0.004
    sigma: float = 0.25          # m/s
    smooth: float = 0.6
    vmax: float = 0.5            # m/s
    rho: float = 0.06            # metres per radian in the pose cost
    w_tip: float = 0.15
    w_effort: float = 0.002
    z_push: float = 0.015
    gain: float = 3.0
    exec_noise: float = 0.0      # std of noise added to the executed joint-delta action (labels stay clean)


# ---------------------------------------------------------------- batched math
def wrap(a):
    return (a + math.pi) % (2 * math.pi) - math.pi


def yaw(q):
    """(B, 4) wxyz -> (B,) yaw."""
    w, x, y, z = q.unbind(-1)
    return torch.atan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))


def rot_error(q_cur, q_ref):
    """Small-angle rotation vectors (B, 3) taking q_cur to q_ref (both wxyz)."""
    w1, x1, y1, z1 = q_cur.unbind(-1)
    w2, x2, y2, z2 = q_ref.unbind(-1)
    w = w2 * w1 + x2 * x1 + y2 * y1 + z2 * z1
    x = -w2 * x1 + x2 * w1 - y2 * z1 + z2 * y1
    y = -w2 * y1 + x2 * z1 + y2 * w1 - z2 * x1
    z = -w2 * z1 - x2 * y1 + y2 * x1 + z2 * w1
    s = torch.where(w < 0, -1.0, 1.0)
    return 2.0 * torch.stack([x * s, y * s, z * s], -1)


def tip_to_t_distance(tip, t_xy, t_th):
    """Distance from tip (B, 2) to the T outline at pose (t_xy, t_th)."""
    poly = T_POLY.to(tip)
    c, s = torch.cos(t_th), torch.sin(t_th)
    px = t_xy[:, None, 0] + c[:, None] * poly[None, :, 0] - s[:, None] * poly[None, :, 1]
    py = t_xy[:, None, 1] + s[:, None] * poly[None, :, 0] + c[:, None] * poly[None, :, 1]
    ax, ay = px, py
    bx, by = px.roll(-1, 1), py.roll(-1, 1)
    abx, aby = bx - ax, by - ay
    t = (((tip[:, None, 0] - ax) * abx + (tip[:, None, 1] - ay) * aby) / (abx ** 2 + aby ** 2)).clamp(0, 1)
    dx = tip[:, None, 0] - (ax + t * abx)
    dy = tip[:, None, 1] - (ay + t * aby)
    return torch.sqrt(dx ** 2 + dy ** 2).min(1).values


class BatchedIK:
    def __init__(self, device):
        import mani_skill
        import pytorch_kinematics as pk
        urdf = Path(mani_skill.__file__).parent / "assets/robots/panda/panda_stick.urdf"
        self.chain = pk.build_serial_chain_from_urdf(urdf.read_bytes(), "panda_hand_tcp").to(dtype=torch.float32,
                                                                                             device=device)

    def joint_delta(self, qpos, dpos, rot_err, damping=0.02):
        J = self.chain.jacobian(qpos)                               # (B, 6, 7)
        target = torch.cat([dpos, rot_err], -1)[..., None]          # (B, 6, 1)
        JJt = J @ J.transpose(1, 2) + (damping ** 2) * torch.eye(6, device=J.device)
        return (J.transpose(1, 2) @ torch.linalg.solve(JJt, target))[..., 0]


# ---------------------------------------------------------------- the expert
class MPPIExpert:
    def __init__(self, env, episodes: int, cfg: MPPIConfig, seed: int = 0):
        self.env, self.u, self.E, self.cfg = env, env.unwrapped, episodes, cfg
        self.N = cfg.samples
        assert self.u.num_envs == self.E * self.N, (self.u.num_envs, self.E, self.N)
        self.device = self.u.device
        self.ik = BatchedIK(self.device)
        self.gen = torch.Generator(device=self.device).manual_seed(seed)
        self.leaders = torch.arange(self.E, device=self.device) * self.N

    # ------------------------------------------------------------ state helpers
    def _snap(self):
        u = self.u
        tcp = u.agent.tcp.pose.raw_pose
        tee = u.tee.pose.raw_pose
        return dict(qpos=u.agent.robot.get_qpos(), qvel=u.agent.robot.get_qvel(), tcp=tcp, tip=tcp[:, :3], tcp_q=tcp[:, 3:],
                    tee=tee, t_xy=tee[:, :2], t_th=yaw(tee[:, 3:]), goal=u.goal_tee.pose.p)

    def _repeat_leaders(self, x):
        if isinstance(x, dict):
            return {k: self._repeat_leaders(v) for k, v in x.items()}
        return x[self.leaders].repeat_interleave(self.N, 0)

    def sync_to_leaders(self):
        """Copy every leader's full simulator state into all copies of its group."""
        state = self.u.get_state_dict()
        self.u.set_state_dict(self._repeat_leaders(state))

    def max_group_divergence(self):
        """Largest difference between any copy and its leader (for the self-check)."""
        s = self._snap()
        diff = (s["tee"] - s["tee"][self.leaders].repeat_interleave(self.N, 0)).abs().max()
        diff_q = (s["qpos"] - s["qpos"][self.leaders].repeat_interleave(self.N, 0)).abs().max()
        return float(diff), float(diff_q)

    # ------------------------------------------------------------ episode
    def reset(self, goals_xy, seed):
        """goals_xy: (E, 2). All copies of an episode get the same goal and the leader's start state."""
        goal_all = torch.as_tensor(goals_xy, dtype=torch.float32, device=self.device).repeat_interleave(self.N, 0)
        self.env.reset(seed=seed, options=dict(goal_xy=goal_all))
        self.sync_to_leaders()
        s = self._snap()
        self.q_ref = s["tcp_q"].clone()
        self.plan = torch.zeros(self.E, self.cfg.horizon, 2, device=self.device)

    def _actions(self, s, vel):
        """Tip velocities (B, 2) -> joint-delta actions (B, 7)."""
        dz = 1.5 * (self.cfg.z_push - s["tip"][:, 2:3])
        d = torch.cat([vel * DT, dz], -1)
        dq = self.ik.joint_delta(s["qpos"], d, 0.5 * rot_error(s["tcp_q"], self.q_ref))
        return (self.cfg.gain * dq / JOINT_DELTA_SCALE).clamp(-1, 1)

    def _noise(self):
        c = self.cfg
        eps = torch.randn(self.E, self.N, c.horizon, 2, device=self.device, generator=self.gen)
        for t in range(1, c.horizon):
            eps[:, :, t] = c.smooth * eps[:, :, t - 1] + math.sqrt(1 - c.smooth ** 2) * eps[:, :, t]
        return eps * c.sigma

    def _cost(self, s):
        c = self.cfg
        e = (s["goal"][:, :2] - s["t_xy"]).norm(dim=-1)
        th = wrap(GOAL_THETA - s["t_th"]).abs()
        pose = torch.sqrt(e ** 2 + (c.rho * th) ** 2)
        return pose + c.w_tip * tip_to_t_distance(s["tip"][:, :2], s["t_xy"], s["t_th"])

    @torch.no_grad()
    def step(self):
        """One real control step for all E episodes. Returns (leader obs (E, 31), clean action (E, 7), success (E,))."""
        c, u, E, N = self.cfg, self.u, self.E, self.N
        steps_before = u._elapsed_steps.clone()
        self.sync_to_leaders()
        saved = u.get_state_dict()
        s0 = self._snap()
        obs31 = torch.cat([s0["qpos"], s0["qvel"], s0["tcp"], s0["goal"], s0["tee"]], -1)[self.leaders]
        cand = (self.plan[:, None] + self._noise()).clamp(-c.vmax, c.vmax)       # (E, N, H, 2)
        cand[:, 0] = self.plan
        flat = cand.reshape(E * N, c.horizon, 2)
        for t in range(c.horizon):
            s = self._snap()
            u.step(self._actions(s, flat[:, t]))
        cost = self._cost(self._snap()).reshape(E, N) + c.w_effort * (cand ** 2).sum((2, 3))
        w = torch.softmax(-(cost - cost.min(1, keepdim=True).values) / c.temperature, dim=1)
        self.plan = (w[:, :, None, None] * cand).sum(1)
        u.set_state_dict(saved)
        u._elapsed_steps = steps_before
        s = self._snap()
        vel = self.plan[:, 0].repeat_interleave(N, 0)
        act = self._actions(s, vel)
        clean = act[self.leaders].clone()
        if c.exec_noise > 0:
            noise = torch.randn(E, 7, device=self.device, generator=self.gen) * c.exec_noise
            act = (act + noise.repeat_interleave(N, 0)).clamp(-1, 1)
        _, _, _, _, info = u.step(act)
        self.plan = torch.cat([self.plan[:, 1:], self.plan[:, -1:]], 1)
        return obs31, clean, info["success"][self.leaders].clone()


# ---------------------------------------------------------------- pilot runner
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--episodes-per-batch", type=int, default=8)
    ap.add_argument("--batches-per-quadrant", type=int, default=1)
    ap.add_argument("--quadrants", type=int, nargs="+", default=[1, 2, 3, 4])
    ap.add_argument("--samples", type=int, default=256)
    ap.add_argument("--horizon", type=int, default=10)
    ap.add_argument("--config", default="{}", help="JSON overrides for MPPIConfig")
    ap.add_argument("--seed-base", type=int, default=1001)
    ap.add_argument("--sim-backend", default="physx_cuda")
    ap.add_argument("--self-check", action="store_true", help="verify group syncing and exit")
    ap.add_argument("--frames-episodes", type=int, default=2, help="episodes per quadrant to keep frames for (videos)")
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    import gymnasium as gym
    import pusht_vargoal  # noqa: F401
    from pusht_vargoal import sample_goals

    cfg = MPPIConfig(samples=args.samples, horizon=args.horizon, **json.loads(args.config))
    E = args.episodes_per_batch
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    env = gym.make("PushTVarGoal-v1", obs_mode="state_dict", control_mode="pd_joint_delta_pos",
                   sim_backend=args.sim_backend, num_envs=E * cfg.samples, max_episode_steps=10 ** 9)
    expert = MPPIExpert(env, E, cfg, seed=0)

    if args.self_check:
        goals = sample_goals(E, (1, 2, 3, 4)).numpy()
        expert.reset(goals, seed=0)
        print("after reset: max copy-vs-leader difference (T pose, qpos):", expert.max_group_divergence())
        for _ in range(3):
            expert.step()
        expert.sync_to_leaders()
        print("after 3 steps + sync:", expert.max_group_divergence())
        g = expert._snap()["goal"][:, :2]
        print("goals match within groups:", bool(torch.allclose(g, g[expert.leaders].repeat_interleave(cfg.samples, 0))))
        return

    rows, demo_obs, demo_act, demo_succ, frames_all = [], [], [], [], {}
    t_start = time.time()
    for q in args.quadrants:
        for b in range(args.batches_per_quadrant):
            seed = args.seed_base + 1000 * q + 100 * b
            torch.manual_seed(seed)
            goals = sample_goals(E, (q,)).numpy()
            expert.reset(goals, seed=seed)
            succ, obs_l, act_l, frames = [], [], [], []
            t0 = time.time()
            for t in range(100):
                obs31, act, success = expert.step()
                obs_l.append(obs31.cpu().numpy())
                act_l.append(act.cpu().numpy())
                succ.append(success.cpu().numpy())
                if b == 0:
                    s = expert._snap()
                    L = expert.leaders[:args.frames_episodes]
                    frames.append(dict(tip=s["tip"][L].cpu().numpy(), t_xy=s["t_xy"][L].cpu().numpy(),
                                       t_th=s["t_th"][L].cpu().numpy(), goal=s["goal"][L, :2].cpu().numpy(),
                                       success=success[:args.frames_episodes].cpu().numpy()))
            secs = time.time() - t0
            succ = np.stack(succ, 1)                       # (E, 100)
            for e in range(E):
                rows.append(dict(quadrant=q, batch=b, episode=e, success_once=bool(succ[e].any()),
                                 success_at_end=bool(succ[e, -1]),
                                 first_success_step=int(np.argmax(succ[e]) + 1) if succ[e].any() else 0))
            demo_obs.append(np.stack(obs_l, 1))
            demo_act.append(np.stack(act_l, 1))
            demo_succ.append(succ)
            if b == 0:
                frames_all[q] = frames
            print(f"Q{q} batch {b}: success-once {succ.any(1).mean():.0%}  at-end {succ[:, -1].mean():.0%}  "
                  f"{secs / 100:.3f} s/step  ({secs:.0f} s for {E} episodes)", flush=True)
    total = time.time() - t_start
    res = {}
    for q in args.quadrants:
        r = [x for x in rows if x["quadrant"] == q]
        res[f"Q{q}"] = dict(episodes=len(r), success_once=float(np.mean([x["success_once"] for x in r])),
                            success_at_end=float(np.mean([x["success_at_end"] for x in r])))
    n_eps = len(rows)
    summary = dict(config=asdict(cfg), episodes_per_batch=E, num_envs=E * cfg.samples, episodes=n_eps,
                   seconds_total=round(total), seconds_per_episode=round(total / n_eps, 2),
                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", results=res,
                   mean_success_once=float(np.mean([v["success_once"] for v in res.values()])),
                   mean_success_at_end=float(np.mean([v["success_at_end"] for v in res.values()])))
    (out / "summary.json").write_text(json.dumps(dict(summary=summary, episodes=rows), indent=1))
    np.savez_compressed(out / "demos.npz", obs=np.concatenate(demo_obs), actions=np.concatenate(demo_act),
                        success=np.concatenate(demo_succ))
    np.save(out / "frames.npy", np.array(frames_all, dtype=object), allow_pickle=True)
    print(json.dumps(summary))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/pusht_state_bc/pusht_vargoal.py
# Copied unchanged from jimmy-state-rl:state_rl/pusht_vargoal.py at c87d8ae
"""Variable-goal Push-T: ManiSkill PushT-v1 with a different target position per episode.

Registers ``PushTVarGoal-v1``. Import this module before ``gym.make``.

What changes relative to PushT-v1 (ManiSkill 3.0.1, commit a4a4f92):
  * Goal T position is sampled per episode from four quadrants around ``QUAD_ORIGIN``.
    Goal orientation stays fixed at PushT-v1's 5*pi/3.
  * The success check (``pseudo_render_intersection``) uses a goal transform per
    environment, recomputed at every reset. In PushT-v1 it is one fixed matrix, so a
    moved goal would silently be scored against the old target.
  * The T block starts anywhere in ``TEE_BOX`` (independent of the goal quadrant), with
    its distance to the goal limited to ``START_GOAL_DIST``.
  * The stick tip starts at a random point in ``TIP_BOX``, drawn from a precomputed
    library of arm poses (``assets/tip_start_qpos.npz``, built by make_tip_starts.py),
    at least ``TIP_TEE_MIN`` from the block.

What does NOT change: observation (31 numbers, same order; ``extra/goal_pos`` now
varies), action space and controller, 100-step episodes, the dense reward (it already
reads the goal T's live pose), and the success rule (overlap >= 0.90).

Quadrants, with dx = x - QUAD_ORIGIN[0] (positive = away from the robot base) and y:
  Q1: dx > 0, y > 0    Q2: dx < 0, y > 0    Q3: dx < 0, y < 0    Q4: dx > 0, y < 0
A band of ``AXIS_GAP`` around each axis is excluded so neighbouring quadrants do not
share near-identical goals.

Usage:
  env = gym.make("PushTVarGoal-v1", obs_mode="state", control_mode="pd_joint_delta_pos",
                 goal_quadrants=(1, 2, 3, 4))
  env.reset(seed=0)                                    # random goals in those quadrants
  env.reset(options=dict(goal_xy=torch.tensor([[-0.2, 0.1]])))  # fixed goal(s), [b, 2]
  env.unwrapped.goal_quadrant                          # [num_envs] quadrant of current goal
"""
from pathlib import Path

import numpy as np
import torch

from mani_skill.envs.tasks.tabletop.push_t import PushTEnv
from mani_skill.utils.registration import register_env
from mani_skill.utils.structs.pose import Pose

# ---- Workspace (world frame, metres). Robot base is at x = -0.615. ----------------
# Measured stick-tip reach at table height: x in about [-0.40, 0.05], y in about
# [-0.45, 0.45] (see README). Regions below keep >= ~0.1 m margin inside that.
QUAD_ORIGIN = (-0.156, 0.0)          # PushT-v1's goal x; centred in y
GOAL_HALF = (0.10, 0.20)             # goals in origin +- these
AXIS_GAP = (0.02, 0.03)              # excluded band around each quadrant axis
TEE_BOX = ((-0.30, -0.02), (-0.28, 0.28))   # T centre-of-mass start region
TIP_BOX = ((-0.36, 0.00), (-0.30, 0.30))    # stick-tip start region
START_GOAL_DIST = (0.08, 0.30)       # allowed T-start-to-goal distance
TIP_TEE_MIN = 0.15                   # stick tip starts at least this far from the T
N_CANDIDATES = 64                    # rejection-sampling candidates per reset

_QUAD_SIGNS = {1: (1.0, 1.0), 2: (-1.0, 1.0), 3: (-1.0, -1.0), 4: (1.0, -1.0)}
_DEFAULT_TIP_FILE = Path(__file__).parent / "assets" / "tip_start_qpos.npz"


def quadrant_of(goal_xy: torch.Tensor) -> torch.Tensor:
    """Quadrant (1-4) of goal positions [b, 2]."""
    dx = goal_xy[:, 0] - QUAD_ORIGIN[0]
    dy = goal_xy[:, 1] - QUAD_ORIGIN[1]
    q = torch.full_like(dx, 0, dtype=torch.long)
    q[(dx > 0) & (dy > 0)] = 1
    q[(dx < 0) & (dy > 0)] = 2
    q[(dx < 0) & (dy < 0)] = 3
    q[(dx > 0) & (dy < 0)] = 4
    return q


def sample_goals(b: int, quadrants, device=None) -> torch.Tensor:
    """Uniform goal positions [b, 2] in the given quadrants (each quadrant equally likely)."""
    quads = torch.tensor(list(quadrants), device=device)
    pick = quads[torch.randint(len(quads), (b,), device=device)]
    signs = torch.tensor([_QUAD_SIGNS[int(k)] for k in range(1, 5)], device=device)[pick - 1]
    lo = torch.tensor(AXIS_GAP, device=device)
    hi = torch.tensor(GOAL_HALF, device=device)
    mag = lo + torch.rand(b, 2, device=device) * (hi - lo)
    return torch.tensor(QUAD_ORIGIN, device=device) + signs * mag


def _uniform_box(shape, box, device):
    lo = torch.tensor([box[0][0], box[1][0]], device=device)
    hi = torch.tensor([box[0][1], box[1][1]], device=device)
    return lo + torch.rand(*shape, 2, device=device) * (hi - lo)


def _first_valid(valid: torch.Tensor) -> torch.Tensor:
    """Index of the first True per row of [b, K]; 0 if none (vanishingly rare with K=64)."""
    return valid.float().argmax(dim=1)


@register_env("PushTVarGoal-v1", max_episode_steps=100)
class PushTVarGoalEnv(PushTEnv):
    def __init__(self, *args, goal_quadrants=(1, 2, 3, 4), tip_start_file=None, **kwargs):
        assert set(goal_quadrants) <= {1, 2, 3, 4} and len(goal_quadrants) > 0
        self.goal_quadrants = tuple(int(q) for q in goal_quadrants)
        lib = np.load(tip_start_file or _DEFAULT_TIP_FILE)
        self._tip_qpos_np = lib["qpos"]
        self._tip_xy_np = lib["tip_xy"]
        super().__init__(*args, **kwargs)

    def _load_scene(self, options: dict):
        super()._load_scene(options)
        # One goal transform per environment instead of PushT-v1's single matrix.
        self.world_to_goal_trans = self.world_to_goal_trans.unsqueeze(0).repeat(self.num_envs, 1, 1)
        self.goal_quadrant = torch.zeros(self.num_envs, dtype=torch.long, device=self.device)
        self.tip_qpos_lib = torch.as_tensor(self._tip_qpos_np, device=self.device)
        self.tip_xy_lib = torch.as_tensor(self._tip_xy_np, device=self.device)

    def _goal_world_to_goal(self, goal_xy: torch.Tensor) -> torch.Tensor:
        b = goal_xy.shape[0]
        c, s = np.cos(self.goal_z_rot), np.sin(self.goal_z_rot)
        g = torch.zeros(b, 3, 3, device=goal_xy.device)
        g[:, 0, 0], g[:, 0, 1], g[:, 1, 0], g[:, 1, 1] = c, -s, s, c
        g[:, 2, 2] = 1.0
        g[:, 0:2, 2] = goal_xy
        return torch.linalg.inv(g)

    def _initialize_episode(self, env_idx: torch.Tensor, options: dict):
        # Parent sets up the table, default goal/T and robot rest pose; we override all three.
        super()._initialize_episode(env_idx, options)
        with torch.device(self.device):
            b = len(env_idx)

            # ---- goal ----
            goal_xy = (options or {}).get("goal_xy")
            if goal_xy is None:
                goal_xy = sample_goals(b, self.goal_quadrants, self.device)
            else:
                goal_xy = torch.as_tensor(goal_xy, dtype=torch.float32, device=self.device).reshape(-1, 2)
                if goal_xy.shape[0] == 1 and b > 1:
                    goal_xy = goal_xy.expand(b, 2)
                elif goal_xy.shape[0] == self.num_envs and b < self.num_envs:
                    goal_xy = goal_xy[env_idx]
                assert goal_xy.shape[0] == b, "goal_xy must be [1, 2], [b, 2] or [num_envs, 2]"
            goal_p = torch.zeros(b, 3)
            goal_p[:, :2] = goal_xy
            goal_p[:, 2] = 1e-3
            goal_q = torch.tensor([np.cos(self.goal_z_rot / 2), 0.0, 0.0, np.sin(self.goal_z_rot / 2)],
                                  dtype=torch.float32).expand(b, 4)
            self.goal_tee.set_pose(Pose.create_from_pq(p=goal_p, q=goal_q))
            self.world_to_goal_trans[env_idx] = self._goal_world_to_goal(goal_xy)
            self.goal_quadrant[env_idx] = quadrant_of(goal_xy)

            # ---- T block start: independent of goal quadrant, distance-limited ----
            cand = _uniform_box((b, N_CANDIDATES), TEE_BOX, self.device)          # [b, K, 2]
            d = (cand - goal_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid((d >= START_GOAL_DIST[0]) & (d <= START_GOAL_DIST[1]))
            tee_xy = cand[torch.arange(b), pick]
            tee_p = torch.zeros(b, 3)
            tee_p[:, :2] = tee_xy
            tee_p[:, 2] = 0.04 / 2 + 1e-3
            ang = torch.rand(b) * (2 * torch.pi)
            tee_q = torch.zeros(b, 4)
            tee_q[:, 0], tee_q[:, 3] = (ang / 2).cos(), (ang / 2).sin()
            self.tee.set_pose(Pose.create_from_pq(p=tee_p, q=tee_q))

            # ---- stick tip start: library pose away from the T ----
            n_lib = self.tip_xy_lib.shape[0]
            idx = torch.randint(n_lib, (b, N_CANDIDATES))
            dt = (self.tip_xy_lib[idx] - tee_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid(dt >= TIP_TEE_MIN)
            qpos = self.tip_qpos_lib[idx[torch.arange(b), pick]]
            qpos = qpos + torch.randn_like(qpos) * self.robot_init_qpos_noise
            self.agent.reset(qpos)


In [ ]:
# Jimmy's arm-pose library (base64), from jimmy-state-rl at c87d8ae
TIP_LIBRARY = ("...
""")
data = base64.b64decode("".join(TIP_LIBRARY[1].split()))
assert hashlib.sha256(data).hexdigest() == TIP_LIBRARY[0], "tip_start_qpos.npz: SHA-256 mismatch"
(WORK_ROOT / "assets").mkdir(exist_ok=True)
(WORK_ROOT / "assets" / "tip_start_qpos.npz").write_bytes(data)
MPPI_PY = WORK_ROOT / "mppi_gpu.py"
for name in ("mppi_gpu.py", "pusht_vargoal.py"):
    shutil.copy2(WORK_ROOT / name, OUTPUT_ROOT / name)

## 4. Self-check on the GPU

Episodes run as groups of parallel simulator copies. Check that every copy matches its episode's real state after a reset and
after a few steps (both differences should be 0.0), and that copies share their episode's goal. Stop here if not.

In [ ]:
run([TRAIN_PY, MPPI_PY, "--self-check", "--episodes-per-batch", 2, "--samples", 8, "--horizon", 4,
     "--out", OUTPUT_ROOT / "self_check"], cwd=WORK_ROOT)

## 5. Pilot: three search budgets

32 episodes per budget (8 per quadrant). Prints success and measured seconds per control step for each batch of 8 episodes.

In [ ]:
results = {}
for samples, horizon in CONFIGS:
    out = OUTPUT_ROOT / f"N{samples}_H{horizon}"
    run([TRAIN_PY, MPPI_PY, "--episodes-per-batch", EPISODES_PER_BATCH, "--samples", samples, "--horizon", horizon,
         "--quadrants", *QUADRANTS, "--out", out], show=r"^Q\d", cwd=WORK_ROOT)
    results[(samples, horizon)] = json.loads((out / "summary.json").read_text())["summary"]

## 6. Summary and cost extrapolation

In [ ]:
print(f"{'N x H':>9} | {'once':>5} {'end':>5} | {'s/episode':>9} | {'4,096 demos':>11} | {'16,384 demos':>12} | per quadrant (once/end)")
for (samples, horizon), s in results.items():
    per_ep = s["seconds_per_episode"]
    quads = "  ".join(f"Q{q[1]} {v['success_once']:.0%}/{v['success_at_end']:.0%}" for q, v in s["results"].items())
    print(f"{samples:>4} x {horizon:<3} | {s['mean_success_once']:>5.0%} {s['mean_success_at_end']:>5.0%} | {per_ep:>9.1f} | "
          f"{per_ep * 4096 / 3600:>9.1f} h | {per_ep * 16384 / 3600:>10.1f} h | {quads}")
print("\nGPU:", next(iter(results.values()))["gpu"], "| episodes run in lockstep:", EPISODES_PER_BATCH,
      "| cost scales roughly with N x H and improves with more episodes per batch if GPU memory allows.")
(OUTPUT_ROOT / "pilot_summary.json").write_text(json.dumps({f"N{k[0]}_H{k[1]}": v for k, v in results.items()}, indent=1))

## 7. Videos of the best setting (top-down, drawn from the simulator state each step)

In [ ]:
import numpy as np, imageio.v2 as imageio, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from IPython.display import Video, display

T_POLY = np.array([(-0.1, -0.0625), (0.1, -0.0625), (0.1, -0.0125), (0.025, -0.0125),
                   (0.025, 0.1375), (-0.025, 0.1375), (-0.025, -0.0125), (-0.1, -0.0125)])
GOAL_THETA = 5 / 3 * np.pi

def poly(xy, th):
    c, s = np.cos(th), np.sin(th)
    return T_POLY @ np.array([[c, -s], [s, c]]).T + xy

best = max(results, key=lambda k: (results[k]["mean_success_once"], -results[k]["seconds_per_episode"]))
frames_all = np.load(OUTPUT_ROOT / f"N{best[0]}_H{best[1]}" / "frames.npy", allow_pickle=True).item()
video_dir = OUTPUT_ROOT / "videos"
video_dir.mkdir(exist_ok=True)
for q, frames in frames_all.items():
    for e in range(len(frames[0]["t_th"])):
        path = video_dir / f"N{best[0]}_H{best[1]}_Q{q}_ep{e}.mp4"
        writer = imageio.get_writer(path, fps=20, codec="libx264", quality=7, macro_block_size=1)
        trail = []
        for t, f in enumerate(frames):
            fig, ax = plt.subplots(figsize=(4.6, 4.6), dpi=100)
            ax.set_facecolor("#f6f6f2")
            ax.add_patch(Polygon(poly(f["goal"][e], GOAL_THETA), closed=True, fc="#cfcfcf", ec="#7a7a7a", ls="--"))
            ok = bool(f["success"][e])
            ax.add_patch(Polygon(poly(f["t_xy"][e], f["t_th"][e]), closed=True, fc="#2e9e4f" if ok else "#c8453a",
                                 ec="#333", alpha=0.9))
            trail.append(f["tip"][e][:2])
            tr = np.array(trail[-40:])
            ax.plot(tr[:, 0], tr[:, 1], color="#1f6feb", lw=1, alpha=0.6)
            ax.plot(*f["tip"][e][:2], "o", color="#1f6feb")
            ax.set_xlim(-0.45, 0.15), ax.set_ylim(-0.38, 0.38), ax.set_aspect("equal"), ax.set_xticks([]), ax.set_yticks([])
            ax.set_title(f"MPPI N{best[0]} H{best[1]}  Q{q} ep{e}  step {t + 1}/100{'  SUCCESS' if ok else ''}", fontsize=8)
            fig.tight_layout(); fig.canvas.draw()
            writer.append_data(np.asarray(fig.canvas.buffer_rgba())[..., :3]); plt.close(fig)
        writer.close()
        print("saved", path)
for path in sorted(video_dir.glob("*.mp4"))[:4]:
    display(Video(str(path), embed=True, width=360))

## 8. What to send back

`pilot_summary.json` and the `videos/` folder in this session's Drive folder (or a screenshot of the section 6 table).
Decision rule: if a budget reaches about 80% success-once in every quadrant at an affordable cost, use it to generate the
BC demonstrations (with action-noise injection, as in run 4). If not, the next lever is a larger budget or more episodes per
batch; the table shows how cost scales.